# Experiment 3 — reference training profile, our model

This experiment keeps the verified Experiment 2 winning architecture unchanged and trains it with the optimization profile from [Michedev/flow-matching-mnist](https://github.com/Michedev/flow-matching-mnist): conditional optimal-transport velocity MSE, Adam, learning rate `1e-4`, batch size 32, and ten epoch-equivalents.

The reference repository does not publish a numeric train or validation loss. Our measurable target is therefore to beat Experiment 2's verified best validation loss: **0.124551**.

In [ ]:
import os
from pathlib import Path

# Jupyter may start in experiments/. Move to the repository root before imports.
project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
os.chdir(project_root)

import torch
import wandb

from datasets.mnist import MNISTSampler
from models.config import load_config
from models.flow import FlowModel
from training.path import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from training.trainer import FlowTrainer, model_size_b

CONFIG_PATH = "configs/experiment_3.yaml"
cfg = load_config(CONFIG_PATH)
persistent_root = Path(
    os.getenv("DIFFUSION_DATA_ROOT", "/workspace-global/Diffusion-data")
)
data_root = persistent_root / "datasets" / "mnist"
checkpoints_dir = persistent_root / "checkpoints" / "experiment_3"
samples_dir = persistent_root / "samples" / "experiment_3"
wandb_dir = Path("/workspace/wandb")
for path in (data_root, checkpoints_dir, samples_dir, wandb_dir):
    path.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Experiment 3 is intended to run on a CUDA GPU")
torch.set_float32_matmul_precision("high")
torch.manual_seed(cfg["data"]["seed"])
print("device", device, torch.cuda.get_device_name(0))
print("target validation loss", cfg["source"]["target_val_loss"])

In [ ]:
def make_path(split: str) -> GaussianConditionalProbabilityPath:
    return GaussianConditionalProbabilityPath(
        p_data=MNISTSampler(
            root=str(data_root),
            split=split,
            val_size=cfg["data"]["val_size"],
            seed=cfg["data"]["seed"],
        ),
        p_simple_shape=[1, cfg["data"]["image_size"], cfg["data"]["image_size"]],
        alpha=LinearAlpha(),
        beta=LinearBeta(),
    ).to(device)


train_path = make_path("train")
val_path = make_path("val")
model = FlowModel.from_config(CONFIG_PATH).to(device)
trainer = FlowTrainer(path=train_path, model=model, val_path=val_path)
print(f"model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"model size: {model_size_b(model) / 1024**2:.2f} MiB")

In [ ]:
wandb.login()
train_cfg = cfg["training"]
target_val_loss = cfg["source"]["target_val_loss"]

with wandb.init(
    project="mnist-flow-matching",
    dir=str(wandb_dir),
    name="experiment-3-reference-training-profile",
    tags=["experiment-3", "reference-training-profile"],
    config=cfg,
) as run:
    checkpoint_path = checkpoints_dir / f"{run.id}.pt"
    history = trainer.train(
        num_steps=train_cfg["num_steps"],
        device=device,
        lr=train_cfg["learning_rate"],
        optimizer_name=train_cfg["optimizer"],
        weight_decay=train_cfg["weight_decay"],
        max_grad_norm=train_cfg["max_grad_norm"],
        batch_size=cfg["data"]["batch_size"],
        ckpt_path=checkpoint_path,
        checkpoint_every=train_cfg["checkpoint_every"],
        val_every=train_cfg["val_every"],
        val_batches=train_cfg["val_batches"],
        plot_every=train_cfg["plot_every"],
        n_plot_images=cfg["sampling"]["num_samples"],
        n_plot_steps=20,
        samples_dir=samples_dir,
        show_plots=False,
        wandb_run=run,
    )
    best_train_loss = history["train"].min().item()
    best_val_loss = history["val"].min().item()
    run.summary["loss/train_best"] = best_train_loss
    run.summary["loss/val_best"] = best_val_loss
    run.summary["target/val_loss"] = target_val_loss
    run.summary["target/beaten"] = best_val_loss < target_val_loss
    run.summary["checkpoint"] = str(checkpoint_path)
    run.summary["training_steps"] = train_cfg["num_steps"]

print("best training loss", best_train_loss)
print("best validation loss", best_val_loss)
print("beat Experiment 2 target", best_val_loss < target_val_loss)
print("checkpoint", checkpoint_path)

In [ ]:
import torch.nn.functional as F
from torchvision.utils import save_image

from sampling.ode import EulerSimulator, FlowODE

num_samples = cfg["sampling"]["num_samples"]
ode_steps = cfg["sampling"]["ode_steps"]
generator = torch.Generator(device=device).manual_seed(cfg["data"]["seed"])
noise = torch.randn(num_samples, 1, 32, 32, generator=generator, device=device)
ts = torch.linspace(0, 1, ode_steps + 1, device=device).expand(num_samples, -1)
model.eval()
samples = EulerSimulator(FlowODE(model)).simulate(noise, ts).clamp(-1, 1).cpu()

generation_dir = samples_dir / "final"
individual_dir = generation_dir / "individual"
individual_dir.mkdir(parents=True, exist_ok=True)
torch.save(samples, generation_dir / "samples.pt")
display_samples = F.interpolate(samples, size=(256, 256), mode="nearest")
save_image(
    display_samples,
    generation_dir / "grid_5x5.png",
    nrow=5,
    normalize=True,
    value_range=(-1, 1),
    padding=4,
    pad_value=1,
)
for index, sample in enumerate(display_samples):
    save_image(
        sample,
        individual_dir / f"sample_{index + 1:02d}.png",
        normalize=True,
        value_range=(-1, 1),
    )

print("grid", generation_dir / "grid_5x5.png")